# Módulo 01: Sistemas de Ecuaciones Lineales y Teorema de Rouché-Frobenius

> **Institución:** Universidad San Sebastián (USS) — Sede Patagonia  
> **Carrera:** Ingeniería Civil Informática  
> **Asignatura:** Álgebra Lineal  
> **Docente:** Carol Asencio González  
> **Entorno:** Python 3, SymPy, NumPy, Matplotlib 3D, ipywidgets  

---

### Trazabilidad de Fuentes
[Cátedra USS — Diapositivas Docente] | [Texto Guía — Stanley Grossman] | [Computación Científica]

- **[Cátedra USS — Diapositivas Docente]:** Operaciones Elementales por Fila (OEF), Forma Escalonada Reducida por Filas (RREF) y Teorema de Rouché-Frobenius.
- **[Texto Guía — Stanley Grossman]:** Interpretación geométrica de hiperplanos en $\mathbb{R}^3$, caracterización de grados de libertad y unicidad de soluciones.
- **[Computación Científica]:** Cálculo simbólico exacto con SymPy, visualización de planos volumétricos en $\mathbb{R}^3$ y controles interactivos con `ipywidgets`.

---

> [!NOTE]
> Un sistema de ecuaciones lineales $A\mathbf{x} = \mathbf{b}$ de orden $3 \times 3$ representa algebraicamente la intersección de tres planos en el espacio euclidiano tridimensional $\mathbb{R}^3$. El Teorema de Rouché-Frobenius permite diagnosticar rigurosamente la existencia y multiplicidad de soluciones a través de la comparación entre el rango de la matriz de coeficientes $\operatorname{rg}(A)$, el rango de la matriz ampliada $\operatorname{rg}(A \mid \mathbf{b})$ y el número de incógnitas $n$.

## 1. Fundamentos Teóricos

### 1.1 Representación Matricial y Operaciones Elementales por Fila (OEF)
Dado el sistema de 3 ecuaciones lineales con 3 incógnitas:

$$
\begin{cases}
a_{11}x + a_{12}y + a_{13}z = b_1 \\
a_{21}x + a_{22}y + a_{23}z = b_2 \\
a_{31}x + a_{32}y + a_{33}z = b_3
\end{cases}
\iff
\begin{pmatrix}
a_{11} & a_{12} & a_{13} \\
a_{21} & a_{22} & a_{23} \\
a_{31} & a_{32} & a_{33}
\end{pmatrix}
\begin{pmatrix} x \\ y \\ z \end{pmatrix}
=
\begin{pmatrix} b_1 \\ b_2 \\ b_3 \end{pmatrix}
$$

La matriz ampliada asociada es:

$$
(A \mid \mathbf{b}) = 
\left(\begin{array}{ccc|c}
a_{11} & a_{12} & a_{13} & b_1 \\
a_{21} & a_{22} & a_{23} & b_2 \\
a_{31} & a_{32} & a_{33} & b_3
\end{array}\right)
$$

Las Operaciones Elementales por Fila (OEF) preservan invariante el conjunto solución:
1. **Tipo I (Intercambio):** $F_i \leftrightarrow F_j$.
2. **Tipo II (Escalamiento):** $F_i \to c \cdot F_i$ con $c \neq 0$.
3. **Tipo III (Combinación Lineal):** $F_i \to F_i + k \cdot F_j$.

> [!IMPORTANT]
> **Heurística de Mínimos Pasos en Gauss:**  
> - Priorizar intercambios ($F_i \leftrightarrow F_j$) para ubicar pivotes con valor $\pm 1$.
> - Emplear combinaciones cruzadas enteras ($F_i \to a_{kk} F_i - a_{ik} F_k$) para evitar fracciones tempranas.
> - Anular en bloque los elementos superiores e inferiores de la columna pivotante.

---

### 1.2 Teorema de Rouché-Frobenius y Clasificación Geométrica

| Tipo de Sistema | Condición de Rangos | Grados de Libertad | Interpretación Geométrica en $\mathbb{R}^3$ |
|:---|:---:|:---:|:---| 
| **Compatible Determinado (SCD)** | $\operatorname{rg}(A) = \operatorname{rg}(A \mid \mathbf{b}) = 3$ | $0$ | Los 3 planos se cortan en un **único punto** $P_0(x_0, y_0, z_0)$. |
| **Compatible Indeterminado (SCI)** | $\operatorname{rg}(A) = \operatorname{rg}(A \mid \mathbf{b}) = 2$ | $1$ | Los 3 planos se cortan en una **recta común** $L(t)$. |
| **Compatible Indeterminado (SCI)** | $\operatorname{rg}(A) = \operatorname{rg}(A \mid \mathbf{b}) = 1$ | $2$ | Los 3 planos son **coincidentes** en un plano común $\pi$. |
| **Incompatible (SI)** | $\operatorname{rg}(A) < \operatorname{rg}(A \mid \mathbf{b})$ | N/A | **Sin solución.** Planos paralelos disjuntos o prisma triangular hueco. |

## 2. Implementación Computacional

In [ ]:
# Importaciones y configuración científica
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Paleta Institucional Universidad San Sebastián
USS_BLUE = '#00205B'
USS_GOLD = '#D4AF37'
USS_ACCENT_BLUE = '#1E88E5'
USS_ACCENT_GREEN = '#27AE60'
USS_ACCENT_RED = '#C0392B'
USS_DARK_GRAY = '#2C3E50'
USS_LIGHT_GRAY = '#F8F9F9'

sp.init_printing(use_latex='mathjax')
%matplotlib inline
print('Entorno computacional USS configurado correctamente.')

In [ ]:
def analizar_rouche_frobenius(A_list, b_list):
    """Calcula rangos exactos y clasifica el sistema según Rouché-Frobenius."""
    A_sp = sp.Matrix(A_list)
    b_sp = sp.Matrix(b_list)
    Ab_sp = A_sp.col_insert(3, b_sp)
    
    rg_A = int(A_sp.rank())
    rg_Ab = int(Ab_sp.rank())
    n = 3
    
    if rg_A == rg_Ab:
        if rg_A == n:
            tipo = 'SCD'
            nombre = 'Sistema Compatible Determinado'
            desc = 'Solución única: Intersección de los 3 planos en un único punto P0.'
            gl = 0
        else:
            tipo = 'SCI'
            nombre = 'Sistema Compatible Indeterminado'
            gl = n - rg_A
            desc = f'Infinitas soluciones: {gl} grado(s) de libertad.'
    else:
        tipo = 'SI'
        nombre = 'Sistema Incompatible'
        desc = 'Sin solución: Planos paralelos disjuntos o prisma triangular hueco.'
        gl = None
        
    return {
        'A_sp': A_sp,
        'b_sp': b_sp,
        'Ab_sp': Ab_sp,
        'rg_A': rg_A,
        'rg_Ab': rg_Ab,
        'n': n,
        'tipo': tipo,
        'nombre': nombre,
        'descripcion': desc,
        'grados_libertad': gl
    }

def resolver_gauss_jordan_pasos(A_sp, b_sp):
    """Calcula la forma escalonada reducida por filas (RREF) y la solución analítica."""
    Ab = A_sp.col_insert(3, b_sp)
    rref_mat, pivot_cols = Ab.rref()
    symbols = sp.symbols('x y z')
    sol = sp.linsolve((A_sp, b_sp), symbols)
    return Ab, rref_mat, pivot_cols, sol

## 3. Visualización y Discusión Geométrica

In [ ]:
def graficar_sistema_3d(A_np, b_np, rf_info, elev=25, azim=-50):
    """Grafica los 3 planos en R³ con alpha=0.22, contornos edgecolor='gray' y perspectiva elev=25, azim=-50."""
    fig = plt.figure(figsize=(10, 8), facecolor='white')
    ax = fig.add_subplot(111, projection='3d')
    
    x_vals = np.linspace(-5, 5, 35)
    y_vals = np.linspace(-5, 5, 35)
    X, Y = np.meshgrid(x_vals, y_vals)
    
    colores = [USS_BLUE, USS_GOLD, USS_ACCENT_BLUE]
    etiquetas = [
        f"pi1: {A_np[0,0]:.0f}x + {A_np[0,1]:.0f}y + {A_np[0,2]:.0f}z = {b_np[0]:.0f}",
        f"pi2: {A_np[1,0]:.0f}x + {A_np[1,1]:.0f}y + {A_np[1,2]:.0f}z = {b_np[1]:.0f}",
        f"pi3: {A_np[2,0]:.0f}x + {A_np[2,1]:.0f}y + {A_np[2,2]:.0f}z = {b_np[2]:.0f}"
    ]
    
    for i in range(3):
        a, b_c, c = A_np[i]
        d = b_np[i]
        if abs(c) >= 1e-4:
            Z = (d - a * X - b_c * Y) / c
            ax.plot_surface(X, Y, Z, color=colores[i], alpha=0.22, edgecolor='gray', linewidth=0.3)
        elif abs(b_c) >= 1e-4:
            z_vals = np.linspace(-5, 5, 35)
            X_p, Z_p = np.meshgrid(x_vals, z_vals)
            Y_p = (d - a * X_p - c * Z_p) / b_c
            ax.plot_surface(X_p, Y_p, Z_p, color=colores[i], alpha=0.22, edgecolor='gray', linewidth=0.3)
        elif abs(a) >= 1e-4:
            z_vals = np.linspace(-5, 5, 35)
            Y_p, Z_p = np.meshgrid(y_vals, z_vals)
            X_p = (d - b_c * Y_p - c * Z_p) / a
            ax.plot_surface(X_p, Y_p, Z_p, color=colores[i], alpha=0.22, edgecolor='gray', linewidth=0.3)
            
    handles = [
        plt.Rectangle((0, 0), 1, 1, fc=colores[0], alpha=0.4),
        plt.Rectangle((0, 0), 1, 1, fc=colores[1], alpha=0.4),
        plt.Rectangle((0, 0), 1, 1, fc=colores[2], alpha=0.4),
    ]
    labels = list(etiquetas)
    
    if rf_info['tipo'] == 'SCD':
        try:
            sol = np.linalg.solve(A_np, b_np)
            ax.scatter([sol[0]], [sol[1]], [sol[2]], color=USS_ACCENT_RED, s=130, zorder=10,
                       edgecolor='black', linewidth=1.5)
            # Líneas de proyección punteadas a planos coordenados
            ax.plot([sol[0], sol[0]], [sol[1], sol[1]], [-5, sol[2]], color=USS_ACCENT_RED, linestyle=':', alpha=0.8, linewidth=1.3)
            ax.plot([sol[0], sol[0]], [-5, sol[1]], [sol[2], sol[2]], color=USS_ACCENT_RED, linestyle=':', alpha=0.8, linewidth=1.3)
            ax.plot([-5, sol[0]], [sol[1], sol[1]], [sol[2], sol[2]], color=USS_ACCENT_RED, linestyle=':', alpha=0.8, linewidth=1.3)

            # Marcadores en las proyecciones sobre planos coordenados
            ax.scatter([sol[0]], [sol[1]], [-5], color=USS_DARK_GRAY, s=35, alpha=0.7, edgecolors='black', linewidths=0.8)
            ax.scatter([sol[0]], [-5], [sol[2]], color=USS_DARK_GRAY, s=35, alpha=0.7, edgecolors='black', linewidths=0.8)
            ax.scatter([-5], [sol[1]], [sol[2]], color=USS_DARK_GRAY, s=35, alpha=0.7, edgecolors='black', linewidths=0.8)

            ax.text(sol[0] + 0.35, sol[1] + 0.35, sol[2] + 0.35,
                    f"P0({sol[0]:.1f}, {sol[1]:.1f}, {sol[2]:.1f})",
                    color=USS_ACCENT_RED, fontweight='bold', fontsize=10,
                    bbox=dict(boxstyle="round,pad=0.25", fc="white", ec=USS_ACCENT_RED, lw=1.2, alpha=0.9))
            p0_proxy = plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=USS_ACCENT_RED,
                                  markeredgecolor='black', markersize=9)
            handles.append(p0_proxy)
            labels.append(f"Solución única P0({sol[0]:.1f}, {sol[1]:.1f}, {sol[2]:.1f})")
        except Exception:
            pass
    elif rf_info['tipo'] == 'SCI' and rf_info['grados_libertad'] == 1:
        try:
            t = np.linspace(-4, 4, 100)
            p_base = np.linalg.lstsq(A_np, b_np, rcond=None)[0]
            _, _, vh = np.linalg.svd(A_np)
            v_dir = vh[-1]
            line_pts = p_base[:, None] + v_dir[:, None] * t[None, :]
            ax.plot(line_pts[0], line_pts[1], line_pts[2], color=USS_ACCENT_RED, linewidth=3.5)
            
            p_mid = line_pts[:, len(t)//2]
            ax.scatter([p_mid[0]], [p_mid[1]], [p_mid[2]], color=USS_ACCENT_RED, s=80, zorder=10, edgecolor='black', linewidth=1.2)
            ax.plot([p_mid[0], p_mid[0]], [p_mid[1], p_mid[1]], [-5, p_mid[2]], color=USS_ACCENT_RED, linestyle=':', alpha=0.7, linewidth=1.1)
            ax.plot([p_mid[0], p_mid[0]], [-5, p_mid[1]], [p_mid[2], p_mid[2]], color=USS_ACCENT_RED, linestyle=':', alpha=0.7, linewidth=1.1)
            ax.plot([-5, p_mid[0]], [p_mid[1], p_mid[1]], [p_mid[2], p_mid[2]], color=USS_ACCENT_RED, linestyle=':', alpha=0.7, linewidth=1.1)

            line_proxy = plt.Line2D([0], [0], color=USS_ACCENT_RED, lw=3.5)
            handles.append(line_proxy)
            labels.append('Recta de soluciones L(t)')
        except Exception:
            pass
    elif rf_info['tipo'] == 'SI':
        line_colors = [USS_ACCENT_RED, USS_ACCENT_GREEN, USS_GOLD]
        pairs = [(0, 1), (0, 2), (1, 2)]
        pair_names = ['pi1 y pi2', 'pi1 y pi3', 'pi2 y pi3']
        for p_idx, (i, j) in enumerate(pairs):
            n_i = A_np[i]
            n_j = A_np[j]
            v_dir = np.cross(n_i, n_j)
            norm_v = np.linalg.norm(v_dir)
            if norm_v > 1e-4:
                v_dir_unit = v_dir / norm_v
                N_mat = np.vstack([n_i, n_j])
                d_vec = np.array([b_np[i], b_np[j]])
                p_part = np.linalg.pinv(N_mat) @ d_vec
                t_line = np.linspace(-6, 6, 80)
                pts = p_part[:, None] + v_dir_unit[:, None] * t_line[None, :]
                ax.plot(pts[0], pts[1], pts[2], color=line_colors[p_idx], linewidth=2.5)
                line_proxy = plt.Line2D([0], [0], color=line_colors[p_idx], lw=2.5)
                handles.append(line_proxy)
                labels.append(f'Intersección {pair_names[p_idx]}')
                
    title_str = (f"Rouché-Frobenius: {rf_info['nombre']} ({rf_info['tipo']})
"
                 f"rg(A) = {rf_info['rg_A']} | rg(A|b) = {rf_info['rg_Ab']} | n = {rf_info['n']}")
    ax.set_title(title_str, fontsize=11, fontweight='bold', color=USS_BLUE, pad=15)
    ax.set_xlabel('Eje X', fontweight='bold', color=USS_DARK_GRAY)
    ax.set_ylabel('Eje Y', fontweight='bold', color=USS_DARK_GRAY)
    ax.set_zlabel('Eje Z', fontweight='bold', color=USS_DARK_GRAY)
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_zlim(-5, 5)
    ax.view_init(elev=elev, azim=azim)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(0.0, 0.96), fontsize=8.5, framealpha=0.9)
    plt.tight_layout()
    plt.show()

In [ ]:
sistemas_predefinidos = {
    '1. SCD (Solución Única P0)': {
        'A': [[2, 1, -1], [-3, -1, 2], [-2, 1, 2]],
        'b': [8, -11, -3]
    },
    '2. SCI (Infinitas Soluciones — Recta Común)': {
        'A': [[1, 1, 1], [2, -1, 3], [3, 0, 4]],
        'b': [3, 4, 7]
    },
    '3. SI (Incompatible — Prisma Triangular Hueco)': {
        'A': [[1, 1, 1], [1, -1, 2], [2, 0, 3]],
        'b': [1, 2, 5]
    }
}

dropdown_caso = widgets.Dropdown(
    options=list(sistemas_predefinidos.keys()),
    value=list(sistemas_predefinidos.keys())[0],
    description='Sistema:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='60%')
)

slider_elev = widgets.IntSlider(value=25, min=0, max=90, step=5, description='Elevación:')
slider_azim = widgets.IntSlider(value=-50, min=-180, max=180, step=5, description='Azimut:')

out_display = widgets.Output()

def actualizar_simulador(change=None):
    with out_display:
        clear_output(wait=True)
        caso_sel = sistemas_predefinidos[dropdown_caso.value]
        A_val = caso_sel['A']
        b_val = caso_sel['b']
        
        rf = analizar_rouche_frobenius(A_val, b_val)
        Ab_init, rref_mat, pivotes, sol = resolver_gauss_jordan_pasos(rf['A_sp'], rf['b_sp'])
        
        display(HTML(f"""
        <div style="background-color: {USS_LIGHT_GRAY}; padding: 12px; border-left: 5px solid {USS_BLUE}; border-radius: 4px; margin-bottom: 12px;">
            <h3 style="color: {USS_BLUE}; margin: 0;">Diagnóstico: {rf['nombre']} ({rf['tipo']})</h3>
            <p style="margin: 4px 0 0 0; color: {USS_DARK_GRAY}; font-size: 14px;">
                <b>Condición de Rangos:</b> rg(A) = <b>{rf['rg_A']}</b> | rg(A|b) = <b>{rf['rg_Ab']}</b> | n = <b>{rf['n']}</b><br>
                <b>Interpretación:</b> {rf['descripcion']}
            </p>
        </div>
        """))
        
        print('Matriz Ampliada Inicial (A|b):')
        display(Ab_init)
        
        print('Forma Escalonada Reducida por Filas (RREF):')
        display(rref_mat)
        
        print(f'Conjunto Solución Analítico: {sol}')
        
        graficar_sistema_3d(np.array(A_val, dtype=float), np.array(b_val, dtype=float), rf,
                            elev=slider_elev.value, azim=slider_azim.value)

dropdown_caso.observe(actualizar_simulador, names='value')
slider_elev.observe(actualizar_simulador, names='value')
slider_azim.observe(actualizar_simulador, names='value')

display(widgets.VBox([
    dropdown_caso,
    widgets.HBox([slider_elev, slider_azim]),
    out_display
]))

actualizar_simulador()

---

## Criterios de Diagnóstico Rápido para Evaluaciones

> [!EXAMPLE]
> **Checklist de Rouché-Frobenius:**
> 1. **Triángulo de Pivotes No Nulos:** Si tras aplicar OEF se obtienen 3 pivotes distintos de cero en la diagonal de $A$, entonces $\operatorname{rg}(A) = \operatorname{rg}(A \mid \mathbf{b}) = 3 = n$, concluyendo **SCD** (Solución única).
> 2. **Fila Nula Consistente $(0 \ 0 \ 0 \mid 0)$:** Indica que una de las ecuaciones es combinación lineal de las restantes. Al no existir contradicción, $\operatorname{rg}(A) = \operatorname{rg}(A \mid \mathbf{b}) < 3$, concluyendo **SCI** (Infinitas soluciones).
> 3. **Fila Inconsistente $(0 \ 0 \ 0 \mid c)$ con $c \neq 0$:** Representa la proposición contradictoria $0 = c$. En ese punto $\operatorname{rg}(A) < \operatorname{rg}(A \mid \mathbf{b})$, concluyendo **SI** (Sistema Incompatible).